### **Manual labels versus OmniCloudMask**

Upload original images, manual labels and already generated OmniCloudMask predictions.

Use the **same filename stem** in all three uploads.
**Output:** side-by-side masks and errors, per-image CSV, pooled and mean precision/recall/F1/IoU/specificity, a thick-cloud-only sensitivity check, pixel-coverage table, Markdown and HTML report, converted masks and run metadata.

Standard OCM classes: 0 clear, 1 thick cloud, 2 thin cloud, 3 shadow. Here 1+2 = cloud; 0+3 = non-cloud. Documentation: https://omnicloudmask.readthedocs.io/en/latest/

OCM inference requires suitable Red/Green/NIR imagery. This notebook uses exported outputs and original RGB previews only. Do not substitute RGB into a Red/Green/NIR inference workflow.

Avoid JPEG for labels and predictions: lossy compression changes pixel values, and the notebook will reject them rather than guess.

In [ ]:
%pip -q install rasterio

#### **0. Setup**
Creates a fresh, timestamped working folder. To start over (for example to re-upload files), re-run this cell.

In [ ]:
from datetime import datetime
from pathlib import Path
import json, shutil
from PIL import Image
from IPython.display import display
from google.colab import files

PROJECT = Path('/content/omni_eval_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
PROJECT.mkdir(parents=True)
print('Working folder:', PROJECT)

In [ ]:
%%writefile /content/pipeline.py
"""Evaluate exported OmniCloudMask class rasters against manual cloud labels."""
from datetime import datetime, timezone
from pathlib import Path
import csv
import hashlib
import html
import importlib.metadata as importlib_metadata
import io
import json
import platform
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# RGB color in the manual annotation -> label value (255 = ignore)
UNCERTAIN_COLOR = (184, 61, 245)
UNLABELLED_COLOR = (0, 0, 0)
COLORS = {
    (52, 209, 183): 0,           # clear
    (240, 120, 240): 1,          # cloud
    UNCERTAIN_COLOR: 255,        # uncertain
    UNLABELLED_COLOR: 255,       # unlabelled
}

SUPPORTED_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".npy")
PREDICTION_ENCODINGS = {
    "ocm_classes": [0, 1, 2, 3],
    "binary_01": [0, 1],
    "binary_255": [0, 255],
}

METRICS = ("precision", "recall", "f1", "iou", "specificity")
LABELS = {"precision": "Precision", "recall": "Recall", "f1": "F1",
          "iou": "IoU", "specificity": "Specificity"}

# Error-map colors
GREEN, RED, BLUE, GRAY = [0, 190, 0], [230, 40, 40], [40, 120, 255], [160, 160, 160]

# Text that marks settings/notes the user has not filled in yet
PLACEHOLDER_MARKERS = ("fill in", "not yet reviewed",
                       "record number of scenes", "confirm with team")


# --------------------------------------------------------------------------
# Loading
# --------------------------------------------------------------------------
def unpack_upload(upload, target, annotation=False):
    """Save uploaded files (or zip contents) into `target`; return saved paths."""
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    paths = []

    def save(name, content):
        p = Path(name)
        if p.name.startswith(".") or "__MACOSX" in p.parts:
            return
        if p.suffix.lower() not in SUPPORTED_SUFFIXES:
            return
        if annotation and "screenshot" in p.name.lower():
            return
        out = target / p.name
        if out.exists():
            raise ValueError(
                f"Duplicate filename: {p.name}. Re-run the setup cell to get a "
                "fresh working folder, or remove the duplicate files."
            )
        out.write_bytes(content)
        paths.append(out)

    for name, content in upload.items():
        if not name.lower().endswith(".zip"):
            save(name, content)
            continue
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            names = z.namelist()
            has_annotations = annotation and any(
                "/defaultannot/" in "/" + n for n in names
            )
            for member in z.infolist():
                if member.is_dir():
                    continue
                if has_annotations and "/defaultannot/" not in "/" + member.filename:
                    continue
                save(member.filename, z.read(member))

    if not paths:
        raise ValueError("No supported files uploaded.")
    return paths


def load_array(path, palette_to_rgb=False):
    """Return (array, valid_mask, (crs, transform) or None).

    Palette ("P" mode) PNGs hold class indices. They are only expanded to RGB
    when `palette_to_rgb` is True (colored manual masks and RGB previews).
    """
    p = Path(path)
    geo = None

    if p.suffix.lower() == ".npy":
        a = np.load(p, allow_pickle=False)
        valid = None
    elif p.suffix.lower() in (".tif", ".tiff"):
        import rasterio

        with rasterio.open(p) as ds:
            a = ds.read()
            valid = np.all(ds.read_masks() > 0, axis=0)
            geo = (ds.crs, ds.transform)
            # OCM may write class zero as nodata; do not discard all clear pixels.
            if ds.nodata == 0:
                valid = np.ones((ds.height, ds.width), bool)
        a = a[0] if a.shape[0] == 1 else np.moveaxis(a, 0, -1)
    else:
        im = Image.open(p)
        if im.mode == "P" and palette_to_rgb:
            im = im.convert("RGB")
        a = np.array(im)
        valid = None

    if a.ndim == 3 and a.shape[0] == 1:
        a = a[0]
    if a.ndim == 3 and a.shape[-1] == 1:
        a = a[..., 0]
    if valid is None:
        valid = np.ones(a.shape[:2], bool)
    return a, valid, geo


def _same_geo(g1, g2):
    """Same CRS and transform to within 1/1000 of a pixel."""
    if g1[0] != g2[0]:
        return False
    t1 = np.array(tuple(g1[1])[:6], float)
    t2 = np.array(tuple(g2[1])[:6], float)
    tol = 1e-3 * min(abs(t1[0]), abs(t1[4]))
    return bool(np.allclose(t1, t2, rtol=0, atol=tol))


# --------------------------------------------------------------------------
# Label / prediction decoding
# --------------------------------------------------------------------------
def manual_labels(a, mode):
    """Decode manual annotations to 0 clear / 1 cloud / 255 ignore."""
    if mode == "colored":
        if a.ndim != 3 or a.shape[2] not in (3, 4):
            raise ValueError("Colored labels require RGB/RGBA.")
        if a.shape[2] == 4 and not np.all(a[:, :, 3] == 255):
            raise ValueError("Resolve transparent annotation pixels before use.")
        a = a[:, :, :3]
        out = np.full(a.shape[:2], 255, np.uint8)
        known = np.zeros(out.shape, bool)
        for color, value in COLORS.items():
            match = np.all(a == color, axis=2)
            out[match] = value
            known |= match
        if not known.all():
            raise ValueError("Unknown manual colors. Update COLORS explicitly; do not guess.")
        return out

    if a.ndim != 2:
        raise ValueError("Numeric labels must be single-band.")
    if mode == "binary_255":
        if not np.isin(a, [0, 255]).all():
            raise ValueError("Expected 0 clear / 255 cloud.")
        return (a == 255).astype(np.uint8)
    if mode != "binary_ignore" or not np.isin(a, [0, 1, 255]).all():
        raise ValueError("Expected 0 clear / 1 cloud / 255 ignore.")
    return a.astype(np.uint8)


def ignore_breakdown(a, mode):
    """Return (uncertain, unlabelled) boolean masks for the manual labels."""
    if mode == "colored":
        rgb = a[:, :, :3]
        return (np.all(rgb == UNCERTAIN_COLOR, axis=2),
                np.all(rgb == UNLABELLED_COLOR, axis=2))
    if mode == "binary_ignore":
        # Uncertain and unlabelled cannot be told apart in this encoding.
        return np.zeros(a.shape, bool), a == 255
    return np.zeros(a.shape, bool), np.zeros(a.shape, bool)


def binary_prediction(a, mode, cloud_classes=(1, 2)):
    """Convert raw prediction raster to a boolean cloud mask."""
    if a.ndim != 2:
        raise ValueError(
            "Use raw single-band class output, not a colored visualization or screenshot."
        )
    if mode not in PREDICTION_ENCODINGS or not np.isin(a, PREDICTION_ENCODINGS[mode]).all():
        raise ValueError("Prediction values do not match declared encoding.")
    # OCM classes: 1 = thick cloud, 2 = thin cloud
    return np.isin(a, cloud_classes) if mode == "ocm_classes" else a != 0


# --------------------------------------------------------------------------
# Metrics
# --------------------------------------------------------------------------
def ratios(c):
    """Precision, recall, F1, IoU, specificity from confusion counts (None if undefined)."""
    tp, fp, fn, tn = c["tp"], c["fp"], c["fn"], c["tn"]
    pairs = {
        "precision": (tp, tp + fp),
        "recall": (tp, tp + fn),
        "f1": (2 * tp, 2 * tp + fp + fn),
        "iou": (tp, tp + fp + fn),
        "specificity": (tn, tn + fp),
    }
    return {k: (num / den if den else None) for k, (num, den) in pairs.items()}


def metrics(pred, label, valid):
    if pred.shape != label.shape or valid.shape != label.shape:
        raise ValueError("Grid mismatch; do not resize.")
    v = valid & (label != 255)
    g = label == 1
    p = pred.astype(bool)
    masks = {
        "tp": p & g & v,
        "fp": p & ~g & v,
        "fn": ~p & g & v,
        "tn": ~p & ~g & v,
    }
    counts = {k: int(m.sum()) for k, m in masks.items()}
    if not sum(counts.values()):
        raise ValueError("No valid labelled pixels.")
    return counts, ratios(counts)


def _summarize(rows):
    """Pooled (micro), per-image mean (macro) and per-image spread."""
    pooled = {k: sum(r[k] for r in rows) for k in ("tp", "fp", "fn", "tn")}
    macro, spread = {}, {}
    for k in METRICS:
        vals = [r[k] for r in rows if r[k] is not None]
        macro[k] = float(np.mean(vals)) if vals else None
        spread[k] = ({"min": float(np.min(vals)),
                      "median": float(np.median(vals)),
                      "max": float(np.max(vals))} if vals else None)
    return {
        "images": len(rows),
        "counts": pooled,
        "micro": ratios(pooled),
        "macro": macro,
        "per_image_spread": spread,
    }


# --------------------------------------------------------------------------
# Output helpers
# --------------------------------------------------------------------------
def _error_map(pred, lab, v):
    error = np.zeros((*lab.shape, 3), np.uint8)
    error[pred & (lab == 1) & v] = GREEN
    error[pred & (lab == 0) & v] = RED
    error[~pred & (lab == 1) & v] = BLUE
    error[~v] = GRAY
    return error


def _save_panel(rgb, lab, pred, v, key, path):
    titles = [
        "Original",
        "Manual cloud\n(white cloud, black clear, gray ignored)",
        "OmniCloudMask cloud\n(white cloud, black clear, gray ignored)",
        "Green: correct cloud; red: false cloud\nBlue: missed cloud; gray: ignored",
    ]
    images = [
        rgb,
        np.ma.masked_where(~v, lab),
        np.ma.masked_where(~v, pred),
        _error_map(pred, lab, v),
    ]
    # Masked (ignored) pixels must not blend into white cloud.
    cmap = plt.get_cmap("gray").copy()
    cmap.set_bad(np.array(GRAY) / 255)
    fig, axes = plt.subplots(1, 4, figsize=(16, 5.5))
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
        else:
            ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    fig.suptitle(key)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _versions():
    out = {"python": platform.python_version()}
    for pkg in ("numpy", "Pillow", "matplotlib", "rasterio"):
        try:
            out[pkg] = importlib_metadata.version(pkg)
        except Exception:
            out[pkg] = None
    return out


def _fmt(x):
    return "undefined" if x is None else f"{x:.4f}"


def _markdown_table(rows):
    head = ["Image"] + [LABELS[k] for k in METRICS]
    lines = ["| " + " | ".join(head) + " |", "|---" + "|---:" * len(METRICS) + "|"]
    for r in rows:
        cells = [r["image"]] + [_fmt(r[k]) for k in METRICS]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


COVERAGE_COLS = ("evaluated_pixels", "uncertain_pixels", "unlabelled_pixels",
                 "invalid_pixels")


def _coverage_cells(r):
    return ([r["image"]] + [f"{r[k]:,}" for k in COVERAGE_COLS]
            + [f"{r['evaluated_fraction']:.1%}"])


COVERAGE_HEAD = ["Image", "Evaluated", "Uncertain", "Unlabelled",
                 "Invalid source", "Evaluated share"]


def _coverage_markdown(rows):
    lines = ["| " + " | ".join(COVERAGE_HEAD) + " |",
             "|---" + "|---:" * (len(COVERAGE_HEAD) - 1) + "|"]
    lines += ["| " + " | ".join(_coverage_cells(r)) + " |" for r in rows]
    return "\n".join(lines)


def _html_table(head, body_rows):
    esc = html.escape
    out = ["<table><thead><tr>" + "".join(f"<th>{esc(h)}</th>" for h in head)
           + "</tr></thead><tbody>"]
    for cells in body_rows:
        out.append("<tr>" + "".join(f"<td>{esc(str(c))}</td>" for c in cells) + "</tr>")
    out.append("</tbody></table>")
    return "".join(out)


def _metric_html(rows):
    return _html_table(["Image"] + [LABELS[k] for k in METRICS],
                       [[r["image"]] + [_fmt(r[k]) for k in METRICS] for r in rows])


INTERPRETATION = (
    "Use thick and thin cloud as cloud (classes 1 and 2) for this binary cloud "
    "evaluation; the thick-cloud-only table (when present) shows how sensitive the "
    "scores are to that choice. Class 3 is non-cloud here; cloud-shadow accuracy is "
    "not measured. Keep settings fixed for held-out evaluation. No optimal inference "
    "settings are established by evaluating one exported configuration. Review red "
    "regions for false cloud and blue regions for missed cloud; attribute causes only "
    "after inspecting the original imagery.\n\n"
    "Uncertain and unlabelled manual pixels are excluded, and uncertain pixels are "
    "often where model and annotator disagree most, so scores may be optimistic; see "
    "the pixel coverage table. Results apply only to evaluated labelled pixels; "
    "sparse annotations may not represent the entire scene. Pixels are spatially "
    "correlated, so pixel counts overstate how much independent evidence there is; "
    "use the per-image spread rather than treating pooled scores as precise. Confirm "
    "geographic alignment even when dimensions match. Source nodata must be excluded "
    "explicitly (source validity masks) because OCM class 0 can also represent input "
    "nodata; nodata flags on prediction rasters are ignored. This pipeline does not "
    "infer source validity from black RGB.\n"
)

def _build_report(rows, summary, settings, notes, rows_thick=None):
    parts = [
        "# Manual labels versus OmniCloudMask",
        "",
        "## Per-image metrics",
        "",
        _markdown_table(rows),
        "",
        "## Pixel coverage",
        "",
        _coverage_markdown(rows),
        "",
    ]
    if rows_thick:
        parts += ["## Sensitivity: thick cloud only (class 1)", "",
                  _markdown_table(rows_thick), ""]
    parts += [
        "## Overall metrics",
        "",
        json.dumps(summary, indent=2),
        "",
        "Micro pools pixels; macro weights images equally; per_image_spread gives "
        "min/median/max across images. Undefined ratios are excluded from macro "
        "means and spreads.",
        "",
        "## Settings and dataset use",
        "",
        json.dumps(settings or {}, indent=2),
        "",
        "## Visual findings",
        "",
        json.dumps(notes or {}, indent=2),
        "",
        "## Interpretation and recommended settings",
        "",
        INTERPRETATION,
    ]
    return "\n".join(parts)


def _build_html(rows, rows_thick, summary, settings, notes, keys, panels):
    esc = html.escape
    style = (
        "<style>body{font-family:system-ui,sans-serif;max-width:1100px;"
        "margin:2rem auto;padding:0 1rem;line-height:1.5}"
        "table{border-collapse:collapse;margin:.5rem 0}"
        "td,th{border:1px solid #ccc;padding:4px 10px;text-align:right}"
        "td:first-child,th:first-child{text-align:left}"
        "pre{background:#f5f5f5;color:#111;padding:10px;white-space:pre-wrap}"
        "img{max-width:100%}</style>"
    )
    body = [
        "<h1>Manual versus OmniCloudMask</h1>",
        "<h2>Per-image metrics</h2>", _metric_html(rows),
        "<h2>Pixel coverage</h2>",
        _html_table(COVERAGE_HEAD, [_coverage_cells(r) for r in rows]),
    ]
    if rows_thick:
        body += ["<h2>Sensitivity: thick cloud only (class 1)</h2>",
                 _metric_html(rows_thick)]
    body += [
        "<h2>Overall metrics</h2><pre>" + esc(json.dumps(summary, indent=2)) + "</pre>",
        "<h2>Settings and dataset use</h2><pre>"
        + esc(json.dumps(settings or {}, indent=2)) + "</pre>",
        "<h2>Visual findings</h2><pre>" + esc(json.dumps(notes or {}, indent=2)) + "</pre>",
        "<h2>Interpretation and recommended settings</h2><p>"
        + esc(INTERPRETATION.strip()).replace("\n\n", "</p><p>") + "</p>",
    ]
    for key, panel in zip(keys, panels):
        body.append(f'<h2>{esc(key)}</h2><img src="{esc(panel)}">')
    return '<!doctype html><meta charset="utf-8">' + style + "".join(body)

def _index_by_stem(paths):
    d = {}
    for p in paths:
        if p.stem in d:
            raise ValueError("Duplicate image stem: " + p.stem)
        d[p.stem] = p
    return d


def check_no_placeholders(settings, notes):
    """Refuse to finalize a report that still contains template placeholder text."""
    bad = []
    for name, d in (("SETTINGS", settings), ("NOTES", notes)):
        for k, v in (d or {}).items():
            if any(m in str(v).lower() for m in PLACEHOLDER_MARKERS):
                bad.append(f"{name}['{k}']")
    if bad:
        raise ValueError("Replace placeholder text before finalizing: " + ", ".join(bad))

# --------------------------------------------------------------------------
# Main entry point
# --------------------------------------------------------------------------
def run(manual_paths, pred_paths, image_paths, output,
        manual_mode="colored", prediction_mode="ocm_classes",
        settings=None, notes=None, validity_paths=None):
    manuals, preds, images = map(_index_by_stem, (manual_paths, pred_paths, image_paths))
    if not manuals or set(manuals) != set(preds) or set(manuals) != set(images):
        raise ValueError(
            "All three groups must have exactly matching filename stems. "
            f"Manual: {sorted(manuals)}; predictions: {sorted(preds)}; "
            f"images: {sorted(images)}"
        )
    validity = _index_by_stem(validity_paths) if validity_paths else {}
    if validity and set(validity) != set(manuals):
        raise ValueError("Source validity masks must cover every image.")

    out = Path(output)
    out.mkdir(parents=True, exist_ok=False)
    rows, rows_thick, panels, provenance = [], [], [], []
    keys = sorted(manuals)

    for i, key in enumerate(keys):
        a, mv, mg = load_array(manuals[key], palette_to_rgb=(manual_mode == "colored"))
        lab = manual_labels(a, manual_mode)
        pa, _pred_nodata, pg = load_array(preds[key])   # prediction nodata flags are not used
        pred = binary_prediction(pa, prediction_mode)
        rgb, iv, ig = load_array(images[key], palette_to_rgb=True)

        if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:
            raise ValueError(
                "Original previews must be uint8 RGB; upload original RGB "
                "PNG/JPEG or a verified RGB export."
            )
        if rgb.shape[:2] != lab.shape or pred.shape != lab.shape:
            raise ValueError(
                key + ": dimensions differ. Use outputs on the same grid, "
                "without automatic resizing."
            )
        geos = [g for g in (mg, pg, ig) if g is not None and g[0] is not None]
        if geos and any(not _same_geo(g, geos[0]) for g in geos[1:]):
            raise ValueError(key + ": CRS or transform differs.")

        # Validity comes from the manual file, the original image and (optionally)
        # explicit source-validity masks, never from the prediction raster.
        valid = mv & iv
        if validity:
            va, _, vg = load_array(validity[key])
            if va.shape != lab.shape or not np.isin(va, [0, 1]).all():
                raise ValueError("Validity must be a matching 0/1 grid.")
            if (vg is not None and ig is not None
                    and vg[0] is not None and ig[0] is not None
                    and not _same_geo(vg, ig)):
                raise ValueError("Validity georeferencing differs.")
            valid &= va.astype(bool)

        counts, scores = metrics(pred, lab, valid)
        v = valid & (lab != 255)
        unc, unl = ignore_breakdown(a, manual_mode)
        row = dict(
            image=key,
            evaluated_pixels=int(v.sum()),
            uncertain_pixels=int(unc.sum()),
            unlabelled_pixels=int(unl.sum()),
            invalid_pixels=int((~valid & (lab != 255)).sum()),
            ignored_pixels=int((~v).sum()),
            evaluated_fraction=float(v.mean()),
            **counts, **scores,
        )

        # Sensitivity check: thick cloud only (OCM class 1).
        if prediction_mode == "ocm_classes":
            thick = binary_prediction(pa, prediction_mode, cloud_classes=(1,))
            c2, s2 = metrics(thick, lab, valid)
            rows_thick.append(dict(image=key, **c2, **s2))
            row.update({"thick_only_" + k: s2[k] for k in METRICS})
        rows.append(row)

        # Converted masks: manual 0 clear / 255 cloud / 127 ignored; prediction 0 / 255.
        manual_png = np.where(lab == 255, 127, np.where(lab == 1, 255, 0)).astype(np.uint8)
        Image.fromarray(manual_png).save(out / f"{i:03d}_manual.png")
        Image.fromarray(pred.astype(np.uint8) * 255).save(out / f"{i:03d}_prediction.png")
        panel = f"{i:03d}_comparison.png"
        _save_panel(rgb, lab, pred, v, key, out / panel)
        panels.append(panel)

        provenance.append({
            group: {"filename": str(p), "sha256": _sha256(p)}
            for group, p in [("manual", manuals[key]),
                             ("prediction", preds[key]),
                             ("original", images[key])]
        })

    summary = _summarize(rows)
    if rows_thick:
        summary["thick_only_sensitivity"] = _summarize(rows_thick)

    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    with (out / "per_image.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    (out / "run_details.json").write_text(json.dumps(
        dict(created_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"),
             versions=_versions(),
             manual_mode=manual_mode, prediction_mode=prediction_mode,
             saved_mask_encoding=dict(manual="0 clear, 255 cloud, 127 ignored",
                                      prediction="0 non-cloud, 255 cloud"),
             settings=settings, notes=notes, files=provenance),
        indent=2,
    ))

    report = _build_report(rows, summary, settings, notes, rows_thick)
    (out / "comparison_report.md").write_text(report)
    (out / "report.html").write_text(
        _build_html(rows, rows_thick, summary, settings, notes, keys, panels))
    return summary, rows, panels

In [ ]:
import importlib, sys
sys.path.insert(0, '/content')
import pipeline
importlib.reload(pipeline)
shutil.copy('/content/pipeline.py', PROJECT / 'pipeline.py')  # so the final ZIP includes it

####**1. Upload manual masks**
Choose your Truth Masks ZIP or manual label files. The provided colored key is supported. Additional colors must be explicitly mapped in the module.

In [ ]:
manual_paths = pipeline.unpack_upload(files.upload(), PROJECT / 'manual', annotation=True)
print([p.name for p in manual_paths])

####**2. Upload raw OmniCloudMask predictions**
Use single-band class GeoTIFF, NPY or numeric grayscale PNG. NPY shape may be H×W or 1×H×W. Match manual filename stems. Do not upload probability maps or screenshots. An exported binary mask is supported if its encoding is set below.

Nodata flags on prediction rasters are **ignored**; use source validity masks (step 4) for nodata.

In [ ]:
pred_paths = pipeline.unpack_upload(files.upload(), PROJECT / 'predictions')
print([p.name for p in pred_paths])

####**3. Upload original RGB images**

Upload exact original uint8 RGB PNG/JPEG images or verified RGB GeoTIFF exports matching the annotations. Shapes, crops, orientation and pixel grids must match. There is no automatic resizing or reprojection. If OCM ran on a coarser grid, produce a documented aligned output upstream before using this notebook.

In [ ]:
image_paths = pipeline.unpack_upload(files.upload(), PROJECT / 'images')
print([p.name for p in image_paths])

#### **4. Declare encodings, dataset use and inference settings**

`colored` is for your supplied manual masks: teal clear, pink cloud, purple uncertain, black unlabelled. Both uncertain and unlabelled are excluded.

Other manual formats: `binary_ignore` = 0 clear, 1 cloud, 255 ignore; `binary_255` = 0 clear, 255 cloud (no ignore value).

Prediction formats: `ocm_classes` = standard 0/1/2/3; `binary_01` = 0/1; `binary_255` = 0/255. Choose explicitly; the notebook does not guess.

If input imagery has nodata, upload original-grid **0/1 source validity masks** in the next cell. OCM may write nodata as 0, which is also clear; prediction values alone cannot identify it.

In [ ]:
MANUAL_MODE = 'colored'
PREDICTION_MODE = 'ocm_classes'
UPLOAD_SOURCE_VALIDITY = False
# If no validity masks are supplied, confirm that all otherwise labelled pixels have valid source data.
ALL_LABELLED_PIXELS_HAVE_VALID_SOURCE = False
SETTINGS = {
    'dataset_split': 'evaluation — confirm with team',
    'model_version': 'fill in',
    'input_bands': 'fill in Red/Green/NIR band indices',
    'input_resolution_m': 'fill in',
    'inference_parameters': 'fill in parameters used to create these predictions',
    'settings_fixed_before_evaluation': 'fill in True/False',
}


In [ ]:
validity_paths = None
if UPLOAD_SOURCE_VALIDITY:
    validity_paths = pipeline.unpack_upload(files.upload(), PROJECT / 'validity')
else:
    assert ALL_LABELLED_PIXELS_HAVE_VALID_SOURCE, 'Confirm valid source data or upload source validity masks.'


#### **5. Review original images with manual and prediction overlays**
Check matching landmarks and boundaries. Equal dimensions alone do not prove alignment.

Left: original. Middle: manual overlay (white = cloud, black = clear; unlabelled and uncertain are transparent). Right: manual cloud outline (cyan) and OmniCloudMask cloud outline (red) on the original. Outlines that are consistently shifted against each other suggest a grid or orientation problem, not a model error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

originals = {p.stem: p for p in image_paths}
predictions = {p.stem: p for p in pred_paths}
assert len(originals) == len(image_paths), 'Duplicate original stems.'
assert len(predictions) == len(pred_paths), 'Duplicate prediction stems.'
for manual in manual_paths:
    assert manual.stem in originals, 'Missing original: ' + manual.stem
    assert manual.stem in predictions, 'Missing prediction: ' + manual.stem
    a, _, _ = pipeline.load_array(manual, palette_to_rgb=(MANUAL_MODE == 'colored'))
    lab = pipeline.manual_labels(a, MANUAL_MODE)
    rgb, _, _ = pipeline.load_array(originals[manual.stem], palette_to_rgb=True)
    pa, _, _ = pipeline.load_array(predictions[manual.stem])
    pred = pipeline.binary_prediction(pa, PREDICTION_MODE)
    assert rgb.shape[:2] == lab.shape, 'Original and annotation grids differ.'
    assert pred.shape == lab.shape, 'Prediction and annotation grids differ.'
    fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
    ax[0].imshow(rgb); ax[0].set_title('Original')
    ax[1].imshow(rgb)
    ax[1].imshow(np.ma.masked_where(lab == 255, lab), cmap='gray', vmin=0, vmax=1, alpha=.4)
    ax[1].set_title('Manual overlay')
    ax[2].imshow(rgb)
    for mask, color in ((lab == 1, 'cyan'), (pred, 'red')):
        if mask.any() and not mask.all():
            ax[2].contour(mask.astype(float), levels=[.5], colors=color, linewidths=.8)
    ax[2].set_title('Outlines: manual cloud (cyan) vs OCM cloud (red)')
    for x in ax: x.axis('off')
    fig.suptitle(manual.stem); plt.show(); plt.close(fig)

####**6. Confirm alignment and calculate metrics**
Confirm that the prediction exports also share this exact grid. If using georeferenced TIFFs, the pipeline checks available CRS and transforms. For files without georeferencing, alignment is your responsibility.

In [ ]:
ALIGNMENT_CONFIRMED = False
assert ALIGNMENT_CONFIRMED, 'Check original, manual and prediction alignment; then set True.'
OUTPUT = PROJECT / ('results_' + datetime.now().strftime('%H%M%S_%f'))
summary, rows, panels = pipeline.run(manual_paths, pred_paths, image_paths, OUTPUT,
    manual_mode=MANUAL_MODE, prediction_mode=PREDICTION_MODE, settings=SETTINGS,
    validity_paths=validity_paths)
print(json.dumps(summary,indent=2))
for row,panel in zip(rows,panels):
    print(row)
    display(Image.open(OUTPUT / panel))


####**7. Add observed strengths and failures, then finalize the report**
Review error maps: **red** = falsely predicted cloud; **blue** = missed cloud; **green** = correctly detected cloud; **gray** = excluded. Black = correctly identified non-cloud.

Replace the prompts below with image names and visible evidence. A higher precision means fewer false cloud detections; higher recall means fewer missed clouds. Do not claim bright surfaces or thin clouds caused errors without checking the imagery. These labels only measure cloud accuracy, not shadow accuracy.

Keep evaluation images separate from tuning. One exported configuration cannot establish optimal inference settings. If a single training image is used, describe the results as a training-image diagnostic.

Also check the **pixel coverage** table: a large share of uncertain pixels means the scores describe only the easier part of the scene. The **thick-cloud-only** table shows how much your scores depend on counting thin cloud as cloud. The final cell below refuses to run while placeholder text remains.

In [ ]:
NOTES = {
    'strengths': 'Not yet reviewed — enter image names and where cloud detection worked.',
    'failure_cases': 'Not yet reviewed — describe red/blue regions with image names.',
    'recommended_settings': 'Not established — document settings chosen on separate tuning images.',
    'limitations': 'Record number of scenes, label coverage and whether images were used in tuning.',
}
# Refuses to finalize while SETTINGS or NOTES still contain template placeholders.
pipeline.check_no_placeholders(SETTINGS, NOTES)
# Regenerate report with the reviewed notes; use a new directory to preserve the earlier run.
FINAL_OUTPUT = PROJECT / ('final_' + datetime.now().strftime('%H%M%S_%f'))
pipeline.run(manual_paths, pred_paths, image_paths, FINAL_OUTPUT,
    manual_mode=MANUAL_MODE, prediction_mode=PREDICTION_MODE, settings=SETTINGS,
    notes=NOTES, validity_paths=validity_paths)
print('Report:', FINAL_OUTPUT / 'report.html')

####**8. Download**
The ZIP includes inputs, reusable Python pipeline, metrics, report and comparison images. Extract it and open the final report.html with its PNG files beside it.

Make sure to download before the Colab session ends.

In [ ]:
archive = shutil.make_archive('/content/omni_comparison_results', 'zip', PROJECT)
files.download(archive)